In [1]:
from pathlib import Path
from io import BytesIO
import csv
import hashlib
import json
import pandas as pd

PROJECT_ROOT = Path(
    "/home/sagemaker-user/AAI-540-Project/"
    "multilingual-it-ticket-routing-mlops"
)
RAW_DIR = PROJECT_ROOT / "data" / "raw"

project_config = json.loads(
    (PROJECT_ROOT / "project_config.json").read_text(encoding="utf-8")
)

raw_files = sorted(RAW_DIR.glob("*.csv"))
assert len(raw_files) == 5, f"Expected 5 CSV files; found {len(raw_files)}."

raw_inventory = []
raw_frames = {}

for path in raw_files:
    content = path.read_bytes()

    # Detect the CSV separator without changing the original file.
    sample = content[:65536].decode("utf-8-sig")
    separator = csv.Sniffer().sniff(
        sample, delimiters=",;\t|"
    ).delimiter

    frame = pd.read_csv(
        BytesIO(content),
        sep=separator,
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False,
        low_memory=False,
    )

    required_columns = {"subject", "body", "queue", "language"}
    missing_columns = required_columns - set(frame.columns)

    if missing_columns:
        raise ValueError(
            f"{path.name}: missing expected columns {sorted(missing_columns)}"
        )

    language_counts = (
        frame["language"].str.strip().str.lower().value_counts().to_dict()
    )

    raw_frames[path.name] = frame
    raw_inventory.append({
        "filename": path.name,
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
        "separator": separator,
        "rows": len(frame),
        "columns": frame.columns.tolist(),
        "language_counts": language_counts,
        "queue_count": int(frame["queue"].str.strip().nunique()),
    })

    print(f"\nFile: {path.name}")
    print(f"Rows: {len(frame):,} | Columns: {len(frame.columns)}")
    print("Column names:", frame.columns.tolist())
    print("Languages:", language_counts)
    print("Queues:", raw_inventory[-1]["queue_count"])


File: aa_dataset-tickets-multi-lang-5-2-50-version.csv
Rows: 28,587 | Columns: 16
Column names: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']
Languages: {'en': 16338, 'de': 12249}
Queues: 10

File: dataset-tickets-german_normalized.csv
Rows: 2,125 | Columns: 5
Column names: ['subject', 'body', 'queue', 'priority', 'language']
Languages: {'de': 2125}
Queues: 10

File: dataset-tickets-german_normalized_50_5_2.csv
Rows: 13,178 | Columns: 5
Column names: ['subject', 'body', 'queue', 'priority', 'language']
Languages: {'de': 13178}
Queues: 42

File: dataset-tickets-multi-lang-4-20k.csv
Rows: 20,000 | Columns: 15
Column names: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']
Languages: {'en': 11923, 'de': 8077}
Queues: 10

File: dataset-tickets-multi-lang3-4k.csv
Rows: 4,000 | Columns: 17
Col

In [2]:
import re
from collections import defaultdict

master_path = (
    PROJECT_ROOT
    / "Cleaned- Master Data set"
    / "multilingual_ticket_master_cleaned.csv"
)

master_bytes = master_path.read_bytes()
assert (
    hashlib.sha256(master_bytes).hexdigest()
    == project_config["dataset_sha256"]
), "Master dataset fingerprint does not match."

master = pd.read_csv(
    BytesIO(master_bytes),
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)

def normalize_text(value):
    # Used only for comparison; original files remain unchanged.
    return re.sub(r"\s+", " ", str(value)).strip().casefold()

def ticket_key(subject, body):
    return (normalize_text(subject), normalize_text(body))

# Record which source files contain each subject/body pair.
raw_sources_by_ticket = defaultdict(set)
raw_labels_by_ticket = defaultdict(set)

for filename, frame in raw_frames.items():
    for subject, body, queue in frame[
        ["subject", "body", "queue"]
    ].itertuples(index=False, name=None):
        key = ticket_key(subject, body)
        if key == ("", ""):
            continue
        raw_sources_by_ticket[key].add(filename)
        raw_labels_by_ticket[key].add(normalize_text(queue))

master_keys = [
    ticket_key(subject, body)
    for subject, body in master[
        ["subject", "body"]
    ].itertuples(index=False, name=None)
]

matched_mask = [key in raw_sources_by_ticket for key in master_keys]
unmatched_master = master.loc[
    [not matched for matched in matched_mask]
].copy()

source_matches = []
for filename in raw_frames:
    count = sum(
        filename in raw_sources_by_ticket.get(key, set())
        for key in master_keys
    )
    source_matches.append({
        "raw_file": filename,
        "master_tickets_with_matching_text": count,
    })

label_mismatches = sum(
    matched
    and normalize_text(queue) not in raw_labels_by_ticket[key]
    for key, queue, matched in zip(
        master_keys, master["queue"], matched_mask
    )
)

print(f"Master tickets: {len(master):,}")
print(f"Text matched to at least one raw file: {sum(matched_mask):,}")
print(f"Text not matched with this comparison: {len(unmatched_master):,}")
print(f"Matched tickets with no matching raw queue label: {label_mismatches:,}")

display(pd.DataFrame(source_matches))

print("\nSource information recorded in the master:")
display(master["source_dataset"].value_counts())

print(
    "\nSource-match counts can overlap because a ticket may occur "
    "in more than one raw file."
)
print("This comparison checks text and labels; it does not reproduce cleaning.")

Master tickets: 44,278
Text matched to at least one raw file: 42,629
Text not matched with this comparison: 1,649
Matched tickets with no matching raw queue label: 0


,raw_file,master_tickets_with_matching_text
0,aa_dataset-tickets-multi-lang-5-2-50-version.csv,27353
1,dataset-tickets-german_normalized.csv,0
2,dataset-tickets-german_normalized_50_5_2.csv,0
3,dataset-tickets-multi-lang-4-20k.csv,19512
4,dataset-tickets-multi-lang3-4k.csv,3994



Source information recorded in the master:


source_dataset
multi-lang-5-2-50-version                     20278
multi-lang-4-20k                              11692
multi-lang-4-20k;multi-lang-5-2-50-version     8308
multi-lang3-4k                                 4000
Name: count, dtype: int64


Source-match counts can overlap because a ticket may occur in more than one raw file.
This comparison checks text and labels; it does not reproduce cleaning.


In [3]:
# Look for raw tickets sharing either the subject or the body
# with an unmatched master ticket.
raw_by_body = defaultdict(list)
raw_by_subject = defaultdict(list)

for filename, frame in raw_frames.items():
    for subject, body in frame[
        ["subject", "body"]
    ].itertuples(index=False, name=None):
        record = (filename, subject, body)

        normalized_subject = normalize_text(subject)
        normalized_body = normalize_text(body)

        if normalized_body:
            raw_by_body[normalized_body].append(record)
        if normalized_subject:
            raw_by_subject[normalized_subject].append(record)

examples_shown = 0

for _, row in unmatched_master.iterrows():
    candidates = raw_by_body.get(normalize_text(row["body"]), [])
    matching_field = "body"

    if not candidates:
        candidates = raw_by_subject.get(
            normalize_text(row["subject"]), []
        )
        matching_field = "subject"

    if not candidates:
        continue

    filename, raw_subject, raw_body = candidates[0]

    print(f"\nExample {examples_shown + 1}")
    print("Possible raw source:", filename)
    print("Matching field:", matching_field)

    # Show only the field that differs.
    if matching_field == "body":
        print("Raw subject:", repr(raw_subject[:300]))
        print("Master subject:", repr(row["subject"][:300]))
    else:
        print("Raw body:", repr(raw_body[:300]))
        print("Master body:", repr(row["body"][:300]))

    examples_shown += 1
    if examples_shown == 5:
        break

print(f"\nExamples displayed: {examples_shown}")
print("These are candidate matches for inspection, not verified matches.")


Example 1
Possible raw source: dataset-tickets-multi-lang3-4k.csv
Matching field: subject
Raw body: 'Dear Customer Support, I am experiencing a compatibility issue with Google Chrome version 102.0, affecting its performance and security settings. This issue began after installing an extension from your store. Could you assist in resolving this problem? Your prompt help will be greatly appreciated.\\'
Master body: 'Dear Customer Support, I am experiencing a compatibility issue with Google Chrome version 102.0, affecting its performance and security settings. This issue began after installing an extension from your store. Could you assist in resolving this problem? Your prompt help will be greatly appreciated. '

Example 2
Possible raw source: dataset-tickets-multi-lang3-4k.csv
Matching field: subject
Raw body: 'Sehr geehrter Kunde des Tech Online Stores,\\n\\nich schreibe, um technische Unterstützung für meinen Dell XPS 13 9310 anzufordern, der während der Ausführung von Aufgaben unerw

In [4]:
def normalize_linebreaks(value):
    text = str(value)
    # Convert literal backslash-n, backslash-r, and backslash-t to spaces.
    text = re.sub(r"\\[nrt]", " ", text)
    return normalize_text(text)

def normalize_trailing_backslash(value):
    text = normalize_linebreaks(value)
    # Check whether removing trailing backslashes improves matching.
    return re.sub(r"\\+$", "", text).strip()

comparison_results = []

for comparison_name, normalizer in [
    ("Whitespace and case", normalize_text),
    ("Also normalize literal line breaks", normalize_linebreaks),
    ("Also remove trailing backslashes", normalize_trailing_backslash),
]:
    source_lookup = defaultdict(set)
    label_lookup = defaultdict(set)

    for filename, frame in raw_frames.items():
        for subject, body, queue in frame[
            ["subject", "body", "queue"]
        ].itertuples(index=False, name=None):
            key = (normalizer(subject), normalizer(body))
            if key == ("", ""):
                continue
            source_lookup[key].add(filename)
            label_lookup[key].add(normalize_text(queue))

    comparison_keys = [
        (normalizer(subject), normalizer(body))
        for subject, body in master[
            ["subject", "body"]
        ].itertuples(index=False, name=None)
    ]

    matched = [key in source_lookup for key in comparison_keys]

    queue_mismatches = sum(
        is_matched
        and normalize_text(queue) not in label_lookup[key]
        for key, queue, is_matched in zip(
            comparison_keys, master["queue"], matched
        )
    )

    comparison_results.append({
        "comparison": comparison_name,
        "matched_master_tickets": sum(matched),
        "unmatched_master_tickets": len(master) - sum(matched),
        "queue_mismatches_among_matches": queue_mismatches,
        "duplicate_master_keys_after_normalization": (
            len(comparison_keys) - len(set(comparison_keys))
        ),
    })

display(pd.DataFrame(comparison_results))

# Preserve remaining unmatched rows for inspection if needed.
remaining_unmatched = master.loc[
    [not value for value in matched]
].copy()

print("These checks establish normalized text matches, not a full cleaning replay.")

,comparison,matched_master_tickets,unmatched_master_tickets,queue_mismatches_among_matches,duplicate_master_keys_after_normalization
0,Whitespace and case,42629,1649,0,0
1,Also normalize literal line breaks,44278,0,0,0
2,Also remove trailing backslashes,44278,0,0,0


These checks establish normalized text matches, not a full cleaning replay.


In [5]:
import boto3
from datetime import datetime, timezone

config_path = PROJECT_ROOT / "project_config.json"
project_config = json.loads(config_path.read_text(encoding="utf-8"))

bucket = project_config["s3_bucket"]
region = project_config["aws_region"]
s3_client = boto3.client("s3", region_name=region)

# Confirm that uploaded files will receive version IDs.
versioning = s3_client.get_bucket_versioning(Bucket=bucket)
assert versioning.get("Status") == "Enabled", "S3 versioning is not enabled."

# Confirm that all local files are unchanged since inspection.
for item in raw_inventory:
    content = (RAW_DIR / item["filename"]).read_bytes()
    assert hashlib.sha256(content).hexdigest() == item["sha256"], (
        f"File changed since inspection: {item['filename']}"
    )

master_sha = hashlib.sha256(master_path.read_bytes()).hexdigest()
assert master_sha == project_config["dataset_sha256"]

report_dir = PROJECT_ROOT / "reports" / "raw_data"
report_dir.mkdir(parents=True, exist_ok=True)
manifest_path = report_dir / "raw_data_manifest.json"

master_source_files = {
    "aa_dataset-tickets-multi-lang-5-2-50-version.csv",
    "dataset-tickets-multi-lang-4-20k.csv",
    "dataset-tickets-multi-lang3-4k.csv",
}

manifest = {
    "archived_at": datetime.now(timezone.utc).isoformat(),
    "source_url": (
        "https://www.kaggle.com/datasets/"
        "tobiasbueck/multilingual-customer-support-tickets/data"
    ),
    "kaggle_version": None,
    "source_note": (
        "Original files supplied by the team. "
        "The Kaggle release/version number has not been verified."
    ),
    "master_dataset_sha256": master_sha,
    "master_rows": len(master),
    "comparison_results": comparison_results,
    "comparison_limit": (
        "Normalized subject/body matches and queue-label agreement were "
        "checked. This does not reproduce the complete cleaning process "
        "or verify every metadata field."
    ),
    "archive_complete": False,
    "files": [],
}

for item in raw_inventory:
    filename = item["filename"]
    content = (RAW_DIR / filename).read_bytes()
    assert hashlib.sha256(content).hexdigest() == item["sha256"]

    key = f"raw/kaggle/{item['sha256']}/{filename}"

    response = s3_client.put_object(
        Bucket=bucket,
        Key=key,
        Body=content,
        ContentType="text/csv",
        Metadata={"sha256": item["sha256"]},
    )

    version_id = response.get("VersionId")
    assert version_id and version_id != "null", "Missing S3 version ID."

    stored = s3_client.get_object(
        Bucket=bucket,
        Key=key,
        VersionId=version_id,
    )
    try:
        stored_bytes = stored["Body"].read()
    finally:
        stored["Body"].close()

    assert len(stored_bytes) == item["bytes"]
    assert hashlib.sha256(stored_bytes).hexdigest() == item["sha256"], (
        f"Uploaded file verification failed: {filename}"
    )

    manifest["files"].append({
        **item,
        "uri": f"s3://{bucket}/{key}",
        "version_id": version_id,
        "download_verified": True,
        "included_in_master_per_cleaning_notes": filename in master_source_files,
    })

    # Save progress after each verified upload.
    manifest_path.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    print(f"Uploaded and verified: {filename}")

manifest["archive_complete"] = True
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(f"\nVerified raw files: {len(manifest['files'])}")
print(f"Manifest saved: {manifest_path.relative_to(PROJECT_ROOT)}")

Uploaded and verified: aa_dataset-tickets-multi-lang-5-2-50-version.csv
Uploaded and verified: dataset-tickets-german_normalized.csv
Uploaded and verified: dataset-tickets-german_normalized_50_5_2.csv
Uploaded and verified: dataset-tickets-multi-lang-4-20k.csv
Uploaded and verified: dataset-tickets-multi-lang3-4k.csv

Verified raw files: 5
Manifest saved: reports/raw_data/raw_data_manifest.json


In [6]:
# Read the completed archive manifest.
manifest_bytes = manifest_path.read_bytes()
saved_manifest = json.loads(manifest_bytes)

assert saved_manifest["archive_complete"]
assert len(saved_manifest["files"]) == 5
assert all(item["download_verified"] for item in saved_manifest["files"])

manifest_sha = hashlib.sha256(manifest_bytes).hexdigest()
manifest_key = f"raw/manifests/{manifest_sha}.json"

# Prepare the README update before making changes.
readme_path = PROJECT_ROOT / "README.md"
readme_text = readme_path.read_text(encoding="utf-8")

old_description = (
    "The following locations document the cleaned master data and derived\n"
    "datasets. They do not establish that original raw source files have been\n"
    "archived in S3."
)

new_description = (
    "Five original CSV files are archived unchanged under `raw/kaggle/` "
    "in the project S3 bucket. Each uploaded version was downloaded and "
    "verified against its local SHA-256 fingerprint.\n\n"
    "See `raw_data_setup.ipynb` for the archive workflow and "
    "[the raw-data manifest](reports/raw_data/raw_data_manifest.json) "
    "for file locations, S3 version IDs, row counts, columns, and fingerprints.\n\n"
    "Beakal's cleaning notes identify the three multilingual files as the "
    "master's inputs. The two German-only files are archived for reference. "
    "All 44,278 master tickets matched raw subject/body text after comparison "
    "normalization of whitespace, case, and literal line breaks, with no "
    "queue-label disagreements. This verifies normalized text coverage; "
    "it does not reproduce every cleaning step or validate all metadata. "
    "The original Kaggle release number remains unverified.\n\n"
    "The following locations document the cleaned master data and derived "
    "datasets."
)

if old_description in readme_text:
    updated_readme = readme_text.replace(
        old_description, new_description, 1
    )
elif new_description in readme_text:
    updated_readme = readme_text
else:
    raise RuntimeError(
        "README section not found. Check the text before updating it."
    )

# Save the manifest itself in versioned S3 storage.
response = s3_client.put_object(
    Bucket=bucket,
    Key=manifest_key,
    Body=manifest_bytes,
    ContentType="application/json",
    Metadata={"sha256": manifest_sha},
)

manifest_version = response.get("VersionId")
assert manifest_version and manifest_version != "null"

stored = s3_client.get_object(
    Bucket=bucket,
    Key=manifest_key,
    VersionId=manifest_version,
)
try:
    downloaded_manifest = stored["Body"].read()
finally:
    stored["Body"].close()

assert hashlib.sha256(downloaded_manifest).hexdigest() == manifest_sha

# Reload the configuration to preserve other saved project details.
project_config = json.loads(config_path.read_text(encoding="utf-8"))
assert (
    project_config["dataset_sha256"]
    == saved_manifest["master_dataset_sha256"]
)

project_config["raw_data_archive"] = {
    "file_count": 5,
    "raw_prefix_uri": f"s3://{bucket}/raw/kaggle/",
    "local_manifest": "reports/raw_data/raw_data_manifest.json",
    "manifest": {
        "uri": f"s3://{bucket}/{manifest_key}",
        "version_id": manifest_version,
        "sha256": manifest_sha,
    },
    "notebook": "raw_data_setup.ipynb",
}

config_path.write_text(
    json.dumps(project_config, indent=2) + "\n",
    encoding="utf-8",
)
readme_path.write_text(updated_readme, encoding="utf-8")

print("Raw-data manifest saved and verified in S3.")
print("Updated: project_config.json")
print("Updated: README.md")

Raw-data manifest saved and verified in S3.
Updated: project_config.json
Updated: README.md
